# 01 — Explore the data

**Goal:** look at real and fake face photos, count how many of each we have, and check the image sizes.
Nothing is trained here and no project files are changed (apart from adding the dataset in Step 2).

**Dataset:** Kaggle `xhlulu/140k-real-and-fake-faces` — real faces come from FFHQ, fake faces from StyleGAN.
More details in [docs/03_data.md](../docs/03_data.md).

**Labels:** `fake` = 1 (what we want to catch), `real` = 0.

**How to use:** run the cells from top to bottom (Shift+Enter). Works on your own computer and in Google Colab.
No GPU is needed.

## Step 1 — Find the project and data

**What this does, in plain words:**

1. Loads the Python tools we need (paths, charts, images).
2. Finds the project folder (the one that has `src/deepfake_kyc` inside). On Colab the project is not there yet, so it downloads (clones) it from GitHub first.
3. Reads the settings file `configs/base.yaml` to learn where the images should be.
4. Counts the images in each folder and prints `Dataset ready.` or tells you to go to Step 2.

This cell does **not** download the dataset (about 4 GB). The images are never stored in git. They must end up in this layout:

```
<project folder>/data/raw/real_vs_fake/real-vs-fake/
├── train/real/*.jpg   train/fake/*.jpg
├── valid/real/*.jpg   valid/fake/*.jpg
└── test/real/*.jpg    test/fake/*.jpg
```

In [ ]:
# 1. Tools we need
import random
import subprocess
import sys
import zipfile
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

# 2. Find the project folder: look in the current folder and every folder above it.
PROJECT_ROOT = None
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "src" / "deepfake_kyc").is_dir():
        PROJECT_ROOT = folder
        break

# Not found (for example on Colab)? Download the project from GitHub into the current folder.
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd() / "UOT_SCS_3546_041_Deep-Learning"
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", "https://github.com/msondhi/UOT_SCS_3546_041_Deep-Learning.git",
                        str(PROJECT_ROOT)], check=True)
print("Project folder:", PROJECT_ROOT)

# 3. Read the settings file to find where the images should be.
sys.path.insert(0, str(PROJECT_ROOT / "src"))   # lets Python import the project's own code
from deepfake_kyc.config import load_config

cfg = load_config(PROJECT_ROOT / "configs" / "base.yaml")
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"       # put the zip here
DATA_ROOT = PROJECT_ROOT / cfg["data"]["root"]     # folder with train/ valid/ test/
SPLITS = ["train", "valid", "test"]

def list_images(split, class_name):
    """All .jpg files in one folder, e.g. list_images("train", "fake")."""
    return sorted((DATA_ROOT / split / class_name).glob("*.jpg"))

# 4. Count the images. Every split needs both real and fake images.
def report_data_status():
    global DATA_AVAILABLE
    DATA_AVAILABLE = True
    for split in SPLITS:
        n_real = len(list_images(split, "real"))
        n_fake = len(list_images(split, "fake"))
        print(f"{split:6} real={n_real:,}  fake={n_fake:,}")
        if n_real == 0 or n_fake == 0:
            DATA_AVAILABLE = False
    if DATA_AVAILABLE:
        print("Dataset ready.")
    else:
        print("Dataset missing or incomplete: add it with Step 2 below.")

report_data_status()

## Step 2 — Add the dataset (skip if Step 1 printed `Dataset ready.`)

The dataset is too big for GitHub, so every computer (and every new Colab session) needs its own copy.
Get the zip from the [Kaggle page](https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces) (Download button, file `archive.zip`).
Then put it into `data/raw/` in **one** of these ways:

| Where you run | What to do |
|---|---|
| Your own computer | Copy `archive.zip` into the project's `data/raw/` folder with File Explorer / Finder. Skip both 2a cells. |
| Colab, small zip | Run **Step 2a** and pick the zip in the file chooser. |
| Colab, full 4 GB zip | Put `archive.zip` in your Google Drive (`My Drive`), then run **Step 2a (Drive)**. Browser upload of 4 GB is very slow. |

Finally run **Step 2b**: it unpacks the zip into `data/raw/` and checks again.
The Kaggle zip already contains `real_vs_fake/real-vs-fake/train|valid|test/real|fake`, so the images land in the right place by themselves.

On Colab, files disappear when the session ends, so you repeat Step 2 in each new session.

In [ ]:
# Step 2a — Colab only: upload the zip from your computer into data/raw/
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

if "google.colab" in sys.modules:
    from google.colab import files
    uploaded = files.upload()          # a file chooser appears below
    for name, content in uploaded.items():
        (RAW_DATA_DIR / name).write_bytes(content)
        print(f"Saved {name} to {RAW_DATA_DIR}")
else:
    print(f"Not on Colab: copy the zip into {RAW_DATA_DIR} by hand, then run Step 2b.")

In [ ]:
# Step 2a (Drive) — Colab only: copy the zip from Google Drive into data/raw/
import shutil

DRIVE_ZIP = "/content/drive/MyDrive/archive.zip"   # change this if your zip has another name or folder

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")      # asks you to allow access to your Drive
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    print("Copying ... (a few minutes for 4 GB)")
    shutil.copy(DRIVE_ZIP, RAW_DATA_DIR)
    print(f"Copied {DRIVE_ZIP} to {RAW_DATA_DIR}")
else:
    print("Not on Colab: skip this cell.")

In [ ]:
# Step 2b — unpack any zip in data/raw/ and check again
if not DATA_AVAILABLE:
    for zip_path in RAW_DATA_DIR.glob("*.zip"):
        print(f"Unpacking {zip_path.name} ... (the full dataset takes a few minutes)")
        with zipfile.ZipFile(zip_path) as archive:
            archive.extractall(RAW_DATA_DIR)

report_data_status()

## Step 3 — Count real and fake images

**What this does:** counts the real and fake images in each split and draws a bar chart.

**Why:** a big difference between the two bars (class imbalance) or an empty folder would be a problem for training.

- `train` teaches the model, `valid` helps choose the best model, `test` is kept for the final score only.

In [ ]:
if DATA_AVAILABLE:
    real_counts = [len(list_images(split, "real")) for split in SPLITS]
    fake_counts = [len(list_images(split, "fake")) for split in SPLITS]

    for split, n_real, n_fake in zip(SPLITS, real_counts, fake_counts):
        print(f"{split:6} total={n_real + n_fake:,} | real (0)={n_real:,} | fake (1)={n_fake:,}")

    positions = range(len(SPLITS))
    plt.figure(figsize=(6, 3.5))
    plt.bar([x - 0.2 for x in positions], real_counts, width=0.4, label="real (0)")
    plt.bar([x + 0.2 for x in positions], fake_counts, width=0.4, label="fake (1)")
    plt.xticks(positions, SPLITS)
    plt.ylabel("images")
    plt.title("Class balance per split")
    plt.legend()
    plt.show()
else:
    print("Skipped: add the dataset in Step 2 first.")

## Step 4 — Check original image sizes

**What this does:** opens 200 random training images and counts their `(width, height)`.

**Why:** the training code resizes every image to one square size (`224×224`) before the model sees it.
It is good to know the original size first. 200 images are enough to see the pattern without opening all 100,000.

In [ ]:
if DATA_AVAILABLE:
    random.seed(0)   # same random picks every time you run the cell
    train_images = list_images("train", "real") + list_images("train", "fake")
    sample = random.sample(train_images, min(200, len(train_images)))

    sizes = Counter()
    for path in sample:
        with Image.open(path) as image:
            sizes[image.size] += 1

    print(f"Checked {len(sample)} training images.")
    print("(width, height) -> how many:")
    for size, count in sizes.most_common():
        print(f"  {size}: {count}")
    image_size = cfg["data"]["image_size"]
    print(f"\nThe training code resizes every image to {image_size}x{image_size}.")
else:
    print("Skipped: add the dataset in Step 2 first.")

## Step 5 — Look at a few examples

**What this does:** shows 4 random real and 4 random fake training faces side by side.

**Why:** a quick check for broken files or wrong labels. Fakes are often hard to spot by eye —
that is why we need a model. A few pictures say nothing about how well a model will work.

In [ ]:
if DATA_AVAILABLE:
    random.seed(0)
    reals = random.sample(list_images("train", "real"), 4)
    fakes = random.sample(list_images("train", "fake"), 4)
    examples = [(path, "real") for path in reals] + [(path, "fake") for path in fakes]

    fig, axes = plt.subplots(1, len(examples), figsize=(2.2 * len(examples), 2.6))
    for ax, (path, label) in zip(axes, examples):
        with Image.open(path) as image:
            ax.imshow(image.convert("RGB"))
        ax.set_title(label)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped: add the dataset in Step 2 first.")

## Next

[02_degradations_gallery.ipynb](02_degradations_gallery.ipynb) shows what these faces look like after the
robustness degradations (blur, JPEG, downscale, noise, screen capture).

## Quick notes to remember

- `train` teaches the model; `valid` helps choose the model; `test` is held back for the final score.
- `fake = 1` is the positive class and `real = 0` is the negative class.
- The size check looks at only 200 training images, not the whole dataset.
- The example faces are only a sanity check. Do not judge detection accuracy from a few pictures.
- Write the real class counts and image sizes from your run into your report.